# FIFA World Cup 2026 — Match Prediction System

Predicts the remaining matches of the 2026 World Cup (Round of 32 onward) using:

1. **Historical results** (1872–2026) for training
2. **Elo ratings**, scraped from eloratings.net if possible, otherwise computed from scratch
3. **Live 2026 fixtures/results** pulled from the `openfootball/worldcup.json` GitHub feed

Two models are built and compared:
- **Dixon-Coles Poisson model** (primary) — predicts the full scoreline distribution for each match
- **XGBoost classifier** (comparison) — predicts Win/Draw/Loss directly from engineered features

Both are backtested on the actual 2018 and 2022 World Cups before being trusted on 2026, then
used to run a 10,000-iteration Monte Carlo simulation of the rest of the bracket.

> **Note on data sources:** the Kaggle dataset *"International football results from 1872-2024"*
> is itself sourced from, and kept in sync with, the public GitHub repo
> `martj42/international_results`. We pull directly from that repo's raw CSV — same data, no
> Kaggle auth needed. If you'd rather use a Kaggle download you already have locally, see the
> fallback in Section 1.


## 0. Setup

In [1]:
import json
import time
import warnings
from collections import defaultdict, deque, Counter

import numpy as np
import pandas as pd
import requests
from scipy.optimize import minimize
from scipy.special import gammaln
from scipy.stats import poisson
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, log_loss
import xgboost as xgb

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
np.random.seed(42)


## 1. Data loading + cleaning

Three fetches, each wrapped in its own try/except since these are free public endpoints that
can occasionally be flaky or rate-limit. If a fetch fails, re-run the cell — or follow the
printed fallback instructions to load data from a local file instead.

In [2]:
# --- 1a. Historical results: GitHub mirror of the Kaggle "International football results
#          1872-2024" dataset. This repo (martj42/international_results) is the original
#          source the Kaggle dataset packages up, and it is kept current (it already includes
#          2026 World Cup qualifiers and group-stage results), so we use it directly.
HIST_URL = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
SHOOTOUT_URL = "https://raw.githubusercontent.com/martj42/international_results/master/shootouts.csv"

def fetch_csv(url, local_fallback):
    try:
        resp = requests.get(url, timeout=20)
        resp.raise_for_status()
        from io import StringIO
        df = pd.read_csv(StringIO(resp.text))
        print(f"Fetched {url}  ->  {len(df)} rows")
        return df
    except Exception as e:
        print(f"[WARN] Could not fetch {url}: {e}")
        print(f"       Falling back to local file '{local_fallback}'.")
        print("       If that file doesn't exist either: download the Kaggle dataset")
        print("       'International football results from 1872-2024' manually and save its")
        print("       results.csv / shootouts.csv next to this notebook.")
        return pd.read_csv(local_fallback)

hist_raw = fetch_csv(HIST_URL, "results.csv")
shootouts_raw = fetch_csv(SHOOTOUT_URL, "shootouts.csv")
hist_raw.head()


Fetched https://raw.githubusercontent.com/martj42/international_results/master/results.csv  ->  49494 rows
Fetched https://raw.githubusercontent.com/martj42/international_results/master/shootouts.csv  ->  680 rows


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [3]:
# --- 1b. Live 2026 World Cup fixtures/results (group stage + knockout bracket) ---
WC2026_URL = "https://raw.githubusercontent.com/openfootball/worldcup.json/master/2026/worldcup.json"

def fetch_json(url, local_fallback):
    try:
        resp = requests.get(url, timeout=20)
        resp.raise_for_status()
        data = resp.json()
        print(f"Fetched {url}  ->  {len(data.get('matches', []))} matches")
        return data
    except Exception as e:
        print(f"[WARN] Could not fetch {url}: {e}")
        print(f"       Falling back to local file '{local_fallback}'.")
        return json.load(open(local_fallback))

wc2026 = fetch_json(WC2026_URL, "wc2026.json")
print("Rounds present:", sorted(set(m['round'] for m in wc2026['matches'])))


Fetched https://raw.githubusercontent.com/openfootball/worldcup.json/master/2026/worldcup.json  ->  104 matches
Rounds present: ['Final', 'Match for third place', 'Matchday 1', 'Matchday 10', 'Matchday 11', 'Matchday 12', 'Matchday 13', 'Matchday 14', 'Matchday 15', 'Matchday 16', 'Matchday 17', 'Matchday 2', 'Matchday 3', 'Matchday 4', 'Matchday 5', 'Matchday 6', 'Matchday 7', 'Matchday 8', 'Matchday 9', 'Quarter-final', 'Round of 16', 'Round of 32', 'Semi-final']


In [4]:
# --- 1c. Elo ratings: try eloratings.net first; if it's unreachable (it blocks scraping for
#          many sandboxed/cloud IPs) we fall back to computing our own Elo from the historical
#          results using the same published World Football Elo methodology. Either way, the
#          rest of the notebook consumes a single `elo` dict, so this swap is transparent.
ELO_SCRAPE_OK = False
elo_scraped = {}
try:
    resp = requests.get("https://www.eloratings.net/2026_World_Cup", timeout=10)
    resp.raise_for_status()
    # eloratings.net renders its table via JS, so a plain GET usually won't contain the data.
    # We still attempt a very loose parse; if it doesn't yield plausible Elo values we discard it.
    import re
    matches = re.findall(r'"team":"([^"]+)","elo":(\d+)', resp.text)
    if matches:
        elo_scraped = {t: float(e) for t, e in matches}
        ELO_SCRAPE_OK = len(elo_scraped) > 20
except Exception as e:
    print(f"[INFO] eloratings.net scrape not available ({e}). Will compute Elo from scratch instead.")

if ELO_SCRAPE_OK:
    print(f"Scraped {len(elo_scraped)} Elo ratings from eloratings.net")
else:
    print("Computing Elo ratings from the historical results CSV instead (see Section 2).")


Computing Elo ratings from the historical results CSV instead (see Section 2).


In [5]:
# --- 1d. Clean + standardize team names, merge in live 2026 results ---
NAME_FIX = {"USA": "United States", "Bosnia & Herzegovina": "Bosnia and Herzegovina"}
def canon(name):
    return NAME_FIX.get(name, name)

hist = hist_raw.copy()
hist["date"] = pd.to_datetime(hist["date"])
hist = hist.dropna(subset=["home_score", "away_score"]).copy()
# Drop any 2026 World Cup rows from the historical file -- it can lag behind the live bracket
# by a day or two; we rebuild 2026 results from the openfootball feed instead (source of truth).
hist = hist[~((hist["tournament"] == "FIFA World Cup") & (hist["date"] >= "2026-01-01"))]
hist["home_score"] = hist["home_score"].astype(int)
hist["away_score"] = hist["away_score"].astype(int)
hist["home_team"] = hist["home_team"].map(canon)
hist["away_team"] = hist["away_team"].map(canon)

GROUND_COUNTRY = {
    "Mexico City": "Mexico", "Guadalajara (Zapopan)": "Mexico", "Monterrey (Guadalupe)": "Mexico",
    "Toronto": "Canada", "Vancouver": "Canada",
    "Atlanta": "United States", "Boston (Foxborough)": "United States", "Dallas (Arlington)": "United States",
    "Houston": "United States", "Kansas City": "United States", "Los Angeles (Inglewood)": "United States",
    "Miami (Miami Gardens)": "United States", "New York/New Jersey (East Rutherford)": "United States",
    "Philadelphia": "United States", "San Francisco Bay Area (Santa Clara)": "United States", "Seattle": "United States",
}

def resolve_score_and_winner(m):
    s = m.get("score")
    if not s or s.get("ft") is None:
        return None
    t1, t2 = m["team1"], m["team2"]
    h, a = s["ft"]
    if "p" in s:
        ph, pa = s["p"]
        return h, a, "pen", (t1 if ph > pa else t2)
    if "et" in s and s["et"][0] != s["et"][1]:
        eh, ea = s["et"]
        return h, a, "et", (t1 if eh > ea else t2)
    if h == a:
        return h, a, "ft", None
    return h, a, "ft", (t1 if h > a else t2)

wc_played_rows = []
for m in wc2026["matches"]:
    t1, t2 = canon(m["team1"]), canon(m["team2"])
    res = resolve_score_and_winner(m)
    if res is None:
        continue
    h, a, decided_by, winner = res
    ground = m.get("ground", "")
    wc_played_rows.append({
        "date": pd.Timestamp(m["date"]), "home_team": t1, "away_team": t2,
        "home_score": h, "away_score": a, "tournament": "FIFA World Cup",
        "city": ground, "country": "",
        "neutral": GROUND_COUNTRY.get(ground) != t1,   # only false for true host-nation "home" games
    })

wc_played = pd.DataFrame(wc_played_rows)
print(f"Recovered {len(wc_played)} played 2026 World Cup matches from the live feed")

combined = pd.concat([hist, wc_played], ignore_index=True).sort_values("date").reset_index(drop=True)
print("Combined training history:", combined.shape, " | latest match:", combined["date"].max().date())
combined.tail()


Recovered 77 played 2026 World Cup matches from the live feed
Combined training history: (49482, 9)  | latest match: 2026-06-30


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49477,2026-06-28,South Africa,Canada,0,1,FIFA World Cup,Los Angeles (Inglewood),,True
49478,2026-06-29,Netherlands,Morocco,1,1,FIFA World Cup,Monterrey (Guadalupe),,True
49479,2026-06-29,Brazil,Japan,2,1,FIFA World Cup,Houston,,True
49480,2026-06-29,Germany,Paraguay,1,1,FIFA World Cup,Boston (Foxborough),,True
49481,2026-06-30,Ivory Coast,Norway,1,2,FIFA World Cup,Dallas (Arlington),,True


## 2. Feature engineering

We do **one single chronological pass** over `combined` to build every feature. Because each
match's features are computed from a running state that only reflects *earlier* matches, this
is leak-free by construction — useful both for training and later for backtesting (filtering
this same dataframe by date reproduces exactly what was knowable at the time).

Features built:
- **Elo difference** (`elo_diff`) — our own Elo, updated after every match (see assumptions below)
- **Recent form** (`form_diff`) — mean goal difference over each team's last 10 matches
- **Head-to-head record** (`h2h_score_home`, `h2h_gd_home`) — win rate / avg goal diff over the
  last 5 meetings between the two specific teams
- **Rest days** (`rest_home`, `rest_away`) — days since each team's previous match
- **Neutral venue flag** (`neutral`)


In [6]:
# ----------------------------------------------------------------------------------------
# TUNABLE ASSUMPTIONS (Elo engine)
#   START_ELO     : rating assigned the first time a team appears (1500 = standard default)
#   HOME_ADV_ELO  : rating-point bonus added to the home team when the venue isn't neutral
#   K-factor tiers below mirror eloratings.net's competition-weight convention
#   FORM_WINDOW   : "recent form" = mean goal diff over the team's last N matches (spec: 10)
#   H2H_WINDOW    : head-to-head record uses the last N meetings between the two teams
# ----------------------------------------------------------------------------------------
START_ELO = 1500.0
HOME_ADV_ELO = 100.0
FORM_WINDOW = 10
H2H_WINDOW = 5

def k_factor(tournament):
    t = str(tournament).lower()
    if "world cup" in t and "qualification" not in t:
        return 60.0
    if any(s in t for s in ["euro", "copa am", "africa", "asian cup", "gold cup", "nations cup"]) and "qualification" not in t:
        return 50.0
    if "qualification" in t:
        return 40.0
    return 30.0

def goal_diff_multiplier(gd):
    gd = abs(gd)
    if gd <= 1:
        return 1.0
    if gd == 2:
        return 1.5
    return (11 + gd) / 8.0


class MatchHistoryState:
    """Running Elo / recent-form / head-to-head / rest-day state. `get_features` reads
    pre-match state only (no leakage); `update` absorbs the match result afterwards."""

    def __init__(self):
        self.elo = defaultdict(lambda: START_ELO)
        self.form = defaultdict(lambda: deque(maxlen=FORM_WINDOW))
        self.last_played = {}
        self.h2h = defaultdict(lambda: deque(maxlen=H2H_WINDOW))

    def get_features(self, home, away, date, neutral):
        eh, ea = self.elo[home], self.elo[away]
        form_h = float(np.mean(self.form[home])) if len(self.form[home]) else 0.0
        form_a = float(np.mean(self.form[away])) if len(self.form[away]) else 0.0
        rest_h = min((date - self.last_played[home]).days, 365) if home in self.last_played else 365
        rest_a = min((date - self.last_played[away]).days, 365) if away in self.last_played else 365
        key = frozenset((home, away))
        past = self.h2h[key]
        if len(past):
            pts, gd_sum = 0.0, 0.0
            for (_, gd_home_perspective, winner) in past:
                gd_sum += gd_home_perspective
                if winner == "draw":
                    pts += 1
                elif winner == home:
                    pts += 3
            h2h_score = pts / (3 * len(past))
            h2h_gd = gd_sum / len(past)
        else:
            h2h_score, h2h_gd = 0.5, 0.0   # no shared history -> neutral prior
        return {
            "elo_home": eh, "elo_away": ea, "elo_diff": eh - ea,
            "form_home": form_h, "form_away": form_a, "form_diff": form_h - form_a,
            "rest_home": rest_h, "rest_away": rest_a,
            "h2h_score_home": h2h_score, "h2h_gd_home": h2h_gd, "h2h_n": len(past),
        }

    def update(self, home, away, date, neutral, home_score, away_score, tournament):
        eh, ea = self.elo[home], self.elo[away]
        eff_h = eh + (0.0 if neutral else HOME_ADV_ELO)
        exp_home = 1.0 / (1.0 + 10 ** ((ea - eff_h) / 400.0))
        gd = home_score - away_score
        actual_home = 1.0 if gd > 0 else (0.5 if gd == 0 else 0.0)
        k = k_factor(tournament) * goal_diff_multiplier(gd)
        delta = k * (actual_home - exp_home)
        self.elo[home] = eh + delta
        self.elo[away] = ea - delta
        self.form[home].append(gd)
        self.form[away].append(-gd)
        self.last_played[home] = date
        self.last_played[away] = date
        winner = "draw" if gd == 0 else (home if gd > 0 else away)
        self.h2h[frozenset((home, away))].append((date, gd, winner))


def build_features(df):
    state = MatchHistoryState()
    rows = []
    for r in df.itertuples():
        f = state.get_features(r.home_team, r.away_team, r.date, r.neutral)
        rows.append({
            "date": r.date, "home_team": r.home_team, "away_team": r.away_team,
            "home_score": r.home_score, "away_score": r.away_score,
            "tournament": r.tournament, "neutral": r.neutral, **f,
        })
        state.update(r.home_team, r.away_team, r.date, r.neutral, r.home_score, r.away_score, r.tournament)
    out = pd.DataFrame(rows)
    out["result"] = np.where(out.home_score > out.away_score, "H",
                      np.where(out.home_score < out.away_score, "A", "D"))
    return out, state

feat, history_state = build_features(combined)

# If we managed to scrape live eloratings.net values, blend them in as the *current* rating
# (our own incrementally-computed Elo still drives all historical features/training).
if ELO_SCRAPE_OK:
    for team, val in elo_scraped.items():
        history_state.elo[canon(team)] = val
    print("Blended in eloratings.net's current ratings as of today.")

print(feat.shape)
print("\nCurrent Elo top 15 (computed from history through", combined['date'].max().date(), "):")
print(pd.Series(dict(history_state.elo)).sort_values(ascending=False).head(15))
feat.tail()


(49482, 19)

Current Elo top 15 (computed from history through 2026-06-30 ):
Argentina      2241.122468
Spain          2204.539690
France         2189.272299
Brazil         2132.754937
England        2098.743189
Colombia       2097.054202
Portugal       2056.576233
Netherlands    2055.209068
Morocco        2034.370101
Mexico         2026.866335
Germany        2007.787286
Norway         2004.808462
Ecuador        2004.399771
Belgium        1998.236517
Switzerland    1996.843615
dtype: float64


,date,home_team,away_team,home_score,away_score,tournament,neutral,elo_home,elo_away,elo_diff,form_home,form_away,form_diff,rest_home,rest_away,h2h_score_home,h2h_gd_home,h2h_n,result
49477,2026-06-28,South Africa,Canada,0,1,FIFA World Cup,True,1712.363215,1871.173767,-158.810552,-0.2,0.9,-1.1,4,4,1.000000,2.0,1,A
49478,2026-06-29,Netherlands,Morocco,1,1,FIFA World Cup,True,2057.379464,2032.199704,25.179760,1.5,1.6,-0.1,4,5,0.666667,-1.0,3,D
49479,2026-06-29,Brazil,Japan,2,1,FIFA World Cup,True,2111.638598,2005.579191,106.059407,1.3,1.3,0.0,5,4,0.800000,-0.6,5,H
49480,2026-06-29,Germany,Paraguay,1,1,FIFA World Cup,True,2017.245082,1903.850760,113.394323,2.2,0.0,2.2,4,4,0.666667,0.5,2,D
49481,2026-06-30,Ivory Coast,Norway,1,2,FIFA World Cup,True,1888.834171,1982.723797,-93.889625,1.1,0.8,0.3,5,4,0.500000,0.0,0,A


## 3. Models

**Primary: Dixon-Coles Poisson model.** Each team gets an attack strength and a defense
strength; expected goals for the match are `lambda = exp(attack_home + defense_away + home_adv)`
and `mu = exp(attack_away + defense_home)`. A low-score correlation term `rho` (Dixon & Coles,
1997) corrects the independence assumption for 0-0/1-0/0-1/1-1 scorelines, which plain
independent Poisson tends to under/over-predict. Matches are exponentially time-weighted so
recent form matters more than results from a decade ago.

**Comparison: XGBoost classifier.** Same engineered features (Elo diff, form diff, h2h,
rest days, neutral flag) feed a gradient-boosted 3-class (Home/Draw/Away) classifier. This
doesn't model the scoreline at all — just W/D/L — so it's a useful sanity check against the
Poisson model's derived outcome probabilities.


In [7]:
# ----------------------------------------------------------------------------------------
# TUNABLE ASSUMPTIONS (Dixon-Coles)
#   XI               : time-decay rate (per day) on the weighted log-likelihood. Higher = the
#                       model forgets older results faster. 0.0015 ~= half-life of ~17 months.
#   DC_TRAIN_YEARS    : how far back the training window reaches
#   DC_MIN_MATCHES    : a team needs at least this many matches in the window to get its own
#                       attack/defense rating; rarer teams fall back to the league average
#                       (keeps the optimizer's parameter count sane: ~250 teams, ~500 params)
#   MAX_GOALS         : size of the scoreline grid used to integrate W/D/L probabilities
# ----------------------------------------------------------------------------------------
XI = 0.0015
DC_TRAIN_YEARS = 10
DC_MIN_MATCHES = 10
MAX_GOALS = 10

def fit_dixon_coles(feat_df, cutoff_date, train_years=DC_TRAIN_YEARS, min_matches=DC_MIN_MATCHES, xi=XI):
    """Fit attack/defense ratings + home advantage + rho using only matches strictly
    before cutoff_date (critical for leak-free backtesting)."""
    train_start = cutoff_date - pd.Timedelta(days=365 * train_years)
    sub = feat_df[(feat_df.date >= train_start) & (feat_df.date < cutoff_date)].copy()

    c = Counter()
    for t in sub.home_team: c[t] += 1
    for t in sub.away_team: c[t] += 1
    keep = sorted(t for t, v in c.items() if v >= min_matches)
    sub = sub[sub.home_team.isin(keep) & sub.away_team.isin(keep)].reset_index(drop=True)

    team_idx = {t: i for i, t in enumerate(keep)}
    n = len(keep)
    home_i = sub.home_team.map(team_idx).to_numpy()
    away_i = sub.away_team.map(team_idx).to_numpy()
    x = sub.home_score.to_numpy(dtype=float)
    y = sub.away_score.to_numpy(dtype=float)
    days_ago = (cutoff_date - sub.date).dt.days.to_numpy()
    weights = np.exp(-xi * days_ago)
    neutral = sub.neutral.to_numpy()

    # identifiability: the model is invariant to alpha_i += c, beta_i -= c for all teams, so
    # we pin alpha[0] = 0 for an arbitrary reference team and optimize everything else.
    def unpack(p):
        alpha = np.empty(n); alpha[0] = 0.0; alpha[1:] = p[: n - 1]
        beta = p[n - 1 : 2 * n - 1]
        gamma, rho = p[-2], p[-1]
        return alpha, beta, gamma, rho

    def tau(x, y, lam, mu, rho):
        out = np.ones_like(lam)
        m00 = (x == 0) & (y == 0); m01 = (x == 0) & (y == 1)
        m10 = (x == 1) & (y == 0); m11 = (x == 1) & (y == 1)
        out[m00] = 1 - lam[m00] * mu[m00] * rho
        out[m01] = 1 + lam[m01] * rho
        out[m10] = 1 + mu[m10] * rho
        out[m11] = 1 - rho
        return np.clip(out, 1e-6, None)

    def nll(p):
        alpha, beta, gamma, rho = unpack(p)
        home_adv = np.where(neutral, 0.0, gamma)
        lam = np.exp(alpha[home_i] + beta[away_i] + home_adv)
        mu = np.exp(alpha[away_i] + beta[home_i])
        log_tau = np.log(tau(x, y, lam, mu, rho))
        ll = log_tau + x * np.log(lam) - lam - gammaln(x + 1) + y * np.log(mu) - mu - gammaln(y + 1)
        return -np.sum(weights * ll)

    p0 = np.zeros(2 * n - 1 + 2)
    p0[-2], p0[-1] = 0.2, -0.05   # initial guesses for home-advantage / rho
    res = minimize(nll, p0, method="L-BFGS-B", options={"maxiter": 400, "maxfun": 20000})
    alpha, beta, gamma, rho = unpack(res.x)

    return {
        "team_idx": team_idx, "alpha": alpha, "beta": beta, "gamma": gamma, "rho": rho,
        "avg_alpha": float(np.mean(alpha)), "avg_beta": float(np.mean(beta)),
        "n_teams": n, "n_matches": len(sub), "converged": res.success, "nll": res.fun,
    }

def _team_strength(model, team):
    i = model["team_idx"].get(team)
    if i is None:
        return model["avg_alpha"], model["avg_beta"]   # unseen team -> league-average rating
    return model["alpha"][i], model["beta"][i]

def dc_score_grid(model, home, away, neutral, max_goals=MAX_GOALS):
    """Returns (max_goals+1) x (max_goals+1) matrix P(home goals=i, away goals=j)."""
    ah, bh = _team_strength(model, home)
    aa, ba = _team_strength(model, away)
    home_adv = 0.0 if neutral else model["gamma"]
    lam = np.exp(ah + ba + home_adv)
    mu = np.exp(aa + bh)
    rho = model["rho"]
    i = np.arange(max_goals + 1)
    grid = np.outer(poisson.pmf(i, lam), poisson.pmf(i, mu))
    def tau1(x, y):
        if x == 0 and y == 0: return 1 - lam * mu * rho
        if x == 0 and y == 1: return 1 + lam * rho
        if x == 1 and y == 0: return 1 + mu * rho
        if x == 1 and y == 1: return 1 - rho
        return 1.0
    for x in (0, 1):
        for y in (0, 1):
            grid[x, y] *= tau1(x, y)
    grid = np.clip(grid, 0, None)
    grid /= grid.sum()
    return grid, lam, mu

def outcome_probs_and_scoreline(grid):
    # grid[i, j] = P(home=i, away=j); row i = home goals, column j = away goals
    pH = float(np.tril(grid, -1).sum())   # home goals > away goals
    pA = float(np.triu(grid, 1).sum())    # away goals > home goals
    pD = float(np.trace(grid))
    i, j = np.unravel_index(np.argmax(grid), grid.shape)
    return pH, pD, pA, (int(i), int(j))

print("Dixon-Coles model functions defined.")


Dixon-Coles model functions defined.


In [8]:
# ---- quick smoke test: fit on everything up to today, sanity-check a marquee matchup ----
dc_model_today = fit_dixon_coles(feat, pd.Timestamp.today() + pd.Timedelta(days=1))
print(f"Fitted on {dc_model_today['n_matches']} matches, {dc_model_today['n_teams']} teams, "
      f"converged={dc_model_today['converged']}, home_adv(log)={dc_model_today['gamma']:.3f}, rho={dc_model_today['rho']:.3f}")

grid, lam, mu = dc_score_grid(dc_model_today, "Argentina", "Brazil", neutral=True)
pH, pD, pA, score = outcome_probs_and_scoreline(grid)
print(f"Argentina vs Brazil (neutral): lambda={lam:.2f} mu={mu:.2f}  P(H)={pH:.2%} P(D)={pD:.2%} P(A)={pA:.2%}  most likely scoreline {score}")


Fitted on 9337 matches, 239 teams, converged=False, home_adv(log)=0.208, rho=-0.094
Argentina vs Brazil (neutral): lambda=1.30 mu=0.80  P(H)=47.08% P(D)=31.15% P(A)=21.77%  most likely scoreline (1, 0)


In [9]:
# ----------------------------------------------------------------------------------------
# Comparison model: XGBoost multi-class classifier on the same engineered features.
# ----------------------------------------------------------------------------------------
FEATURE_COLS = ["elo_diff", "form_diff", "h2h_score_home", "h2h_gd_home", "h2h_n",
                "rest_home", "rest_away", "neutral"]

def fit_xgb_classifier(feat_df, cutoff_date, train_years=12):
    train_start = cutoff_date - pd.Timedelta(days=365 * train_years)
    sub = feat_df[(feat_df.date >= train_start) & (feat_df.date < cutoff_date)].copy()
    sub["neutral"] = sub["neutral"].astype(int)
    le = LabelEncoder()
    y = le.fit_transform(sub["result"])
    X = sub[FEATURE_COLS]
    clf = xgb.XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective="multi:softprob", num_class=3, eval_metric="mlogloss", reg_lambda=1.0,
    )
    clf.fit(X, y)
    return clf, le

def xgb_predict_row(clf, le, row):
    x = pd.DataFrame([{c: (int(row[c]) if c == "neutral" else row[c]) for c in FEATURE_COLS}])
    proba = clf.predict_proba(x)[0]
    classes = le.inverse_transform(np.arange(len(proba)))
    return dict(zip(classes, proba))

xgb_model_today, xgb_le_today = fit_xgb_classifier(feat, pd.Timestamp.today() + pd.Timedelta(days=1))
print("XGBoost classifier trained. Feature importances:")
print(pd.Series(xgb_model_today.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False))


XGBoost classifier trained. Feature importances:
elo_diff          0.403978
h2h_score_home    0.139703
neutral           0.116033
form_diff         0.088880
rest_away         0.064728
h2h_n             0.064035
h2h_gd_home       0.062802
rest_home         0.059841
dtype: float32


## 4. Backtest on 2018 and 2022 World Cups

Before trusting either model on 2026, we refit each one using **only data available before
that tournament kicked off** and check how well it predicted the actual results. This mirrors
exactly how the models will be used on the live 2026 bracket.

In [10]:
def backtest_world_cup(feat_df, year, dc_train_years=DC_TRAIN_YEARS, xgb_train_years=12):
    wc = feat_df[(feat_df.tournament == "FIFA World Cup") & (feat_df.date.dt.year == year)].copy().reset_index(drop=True)
    if wc.empty:
        raise ValueError(f"No FIFA World Cup matches found for {year} in the feature set.")
    cutoff = wc["date"].min()

    dc_m = fit_dixon_coles(feat_df, cutoff, train_years=dc_train_years)
    xgb_m, le = fit_xgb_classifier(feat_df, cutoff, train_years=xgb_train_years)

    rows = []
    for r in wc.itertuples():
        grid, lam, mu = dc_score_grid(dc_m, r.home_team, r.away_team, r.neutral)
        pH, pD, pA, score = outcome_probs_and_scoreline(grid)
        dc_pred = "H" if score[0] > score[1] else ("A" if score[1] > score[0] else "D")
        xgb_p = xgb_predict_row(xgb_m, le, r._asdict())
        rows.append({
            "date": r.date, "home_team": r.home_team, "away_team": r.away_team, "actual": r.result,
            "dc_pH": pH, "dc_pD": pD, "dc_pA": pA, "dc_pred": dc_pred,
            "xgb_pH": xgb_p["H"], "xgb_pD": xgb_p["D"], "xgb_pA": xgb_p["A"], "xgb_pred": max(xgb_p, key=xgb_p.get),
        })
    res = pd.DataFrame(rows)

    def metrics(prefix):
        acc = accuracy_score(res["actual"], res[f"{prefix}_pred"])
        probs = res[[f"{prefix}_pA", f"{prefix}_pD", f"{prefix}_pH"]].to_numpy()
        ll = log_loss(res["actual"], probs, labels=["A", "D", "H"])
        return acc, ll

    dc_acc, dc_ll = metrics("dc")
    xgb_acc, xgb_ll = metrics("xgb")
    print(f"World Cup {year} backtest ({len(res)} matches):")
    print(f"  Dixon-Coles : accuracy={dc_acc:.3f}  log-loss={dc_ll:.3f}")
    print(f"  XGBoost     : accuracy={xgb_acc:.3f}  log-loss={xgb_ll:.3f}")
    print(f"  (for reference, a model that always predicts the uniform 1/3-1/3-1/3 gets log-loss = {np.log(3):.3f})")
    return res, {"dc_acc": dc_acc, "dc_ll": dc_ll, "xgb_acc": xgb_acc, "xgb_ll": xgb_ll}

backtest_2018, metrics_2018 = backtest_world_cup(feat, 2018)
print()
backtest_2022, metrics_2022 = backtest_world_cup(feat, 2022)


World Cup 2018 backtest (64 matches):
  Dixon-Coles : accuracy=0.438  log-loss=0.952
  XGBoost     : accuracy=0.453  log-loss=1.036
  (for reference, a model that always predicts the uniform 1/3-1/3-1/3 gets log-loss = 1.099)

World Cup 2022 backtest (64 matches):
  Dixon-Coles : accuracy=0.453  log-loss=1.079
  XGBoost     : accuracy=0.484  log-loss=1.062
  (for reference, a model that always predicts the uniform 1/3-1/3-1/3 gets log-loss = 1.099)


In [11]:
backtest_2022[["date", "home_team", "away_team", "actual", "dc_pred", "dc_pH", "dc_pD", "dc_pA", "xgb_pred"]].head(10)


,date,home_team,away_team,actual,dc_pred,dc_pH,dc_pD,dc_pA,xgb_pred
0,2022-11-20,Qatar,Ecuador,A,A,0.163053,0.237253,0.599694,A
1,2022-11-21,United States,Wales,D,D,0.443359,0.346179,0.210462,H
2,2022-11-21,England,Iran,H,H,0.477370,0.325228,0.197402,H
3,2022-11-21,Senegal,Netherlands,A,A,0.136609,0.236935,0.626456,A
4,2022-11-22,France,Australia,H,H,0.524631,0.272164,0.203204,H
5,2022-11-22,Denmark,Tunisia,D,H,0.522813,0.293626,0.183561,H
6,2022-11-22,Argentina,Saudi Arabia,A,H,0.775277,0.191041,0.033682,H
7,2022-11-22,Mexico,Poland,D,D,0.422470,0.286743,0.290787,H
8,2022-11-23,Germany,Japan,A,D,0.369844,0.284763,0.345393,H
9,2022-11-23,Spain,Costa Rica,H,H,0.619795,0.263617,0.116588,H


## 5. Apply the trained model to the remaining 2026 fixtures

We refit both models one more time on **all** data through today (`dc_model_today` /
`xgb_model_today` from Section 3 already do this) and apply them to every *currently scheduled*
knockout match that has two known teams (not yet a `Wxx`/`Lxx` placeholder waiting on an
earlier result). Matches further out in the bracket (Round of 16 onward, where opponents are
still unknown) are handled by the full bracket Monte Carlo simulation in Section 6.

In [12]:
def is_placeholder(token):
    return bool(token) and token[0] in "WL" and token[1:].isdigit()

knockout_matches = [m for m in wc2026["matches"] if "num" in m]
knockout_matches.sort(key=lambda m: m["num"])

pending_named = []
for m in knockout_matches:
    t1, t2 = canon(m["team1"]), canon(m["team2"])
    already_played = m.get("score") is not None and m.get("score", {}).get("ft") is not None
    if already_played or is_placeholder(t1) or is_placeholder(t2):
        continue
    pending_named.append({"num": m["num"], "round": m["round"], "date": m["date"],
                           "team1": t1, "team2": t2, "ground": m.get("ground")})

print(f"{len(pending_named)} upcoming fixtures currently have both teams known:")
for m in pending_named:
    print(f"  #{m['num']} {m['round']:<12} {m['date']}  {m['team1']} vs {m['team2']}  ({m['ground']})")


11 upcoming fixtures currently have both teams known:
  #77 Round of 32  2026-06-30  France vs Sweden  (New York/New Jersey (East Rutherford))
  #79 Round of 32  2026-06-30  Mexico vs Ecuador  (Mexico City)
  #80 Round of 32  2026-07-01  England vs DR Congo  (Atlanta)
  #81 Round of 32  2026-07-01  United States vs Bosnia and Herzegovina  (San Francisco Bay Area (Santa Clara))
  #82 Round of 32  2026-07-01  Belgium vs Senegal  (Seattle)
  #83 Round of 32  2026-07-02  Portugal vs Croatia  (Toronto)
  #84 Round of 32  2026-07-02  Spain vs Austria  (Los Angeles (Inglewood))
  #85 Round of 32  2026-07-02  Switzerland vs Algeria  (Vancouver)
  #86 Round of 32  2026-07-03  Argentina vs Cape Verde  (Miami (Miami Gardens))
  #87 Round of 32  2026-07-03  Colombia vs Ghana  (Kansas City)
  #88 Round of 32  2026-07-03  Australia vs Egypt  (Dallas (Arlington))


In [13]:
def predict_named_fixture(m, dc_model, xgb_model, le, state):
    t1, t2, ground = m["team1"], m["team2"], m["ground"]
    neutral = GROUND_COUNTRY.get(ground) != t1   # only False if t1 is the host nation at home

    grid, lam, mu = dc_score_grid(dc_model, t1, t2, neutral)
    pH, pD, pA, score = outcome_probs_and_scoreline(grid)

    feat_row = state.get_features(t1, t2, pd.Timestamp(m["date"]), neutral)
    feat_row["neutral"] = neutral
    xgb_p = xgb_predict_row(xgb_model, le, feat_row)

    return {
        "match_num": m["num"], "round": m["round"], "date": m["date"],
        "team1": t1, "team2": t2, "neutral_venue": neutral,
        "dc_p_team1_win": pH, "dc_p_draw_reg": pD, "dc_p_team2_win": pA,
        "dc_predicted_scoreline": f"{score[0]}-{score[1]}",
        "dc_lambda_team1": round(lam, 2), "dc_mu_team2": round(mu, 2),
        "xgb_p_team1_win": xgb_p["H"], "xgb_p_draw": xgb_p["D"], "xgb_p_team2_win": xgb_p["A"],
    }

named_predictions = pd.DataFrame([
    predict_named_fixture(m, dc_model_today, xgb_model_today, xgb_le_today, history_state)
    for m in pending_named
])
named_predictions


,match_num,round,date,team1,team2,neutral_venue,dc_p_team1_win,dc_p_draw_reg,dc_p_team2_win,dc_predicted_scoreline,dc_lambda_team1,dc_mu_team2,xgb_p_team1_win,xgb_p_draw,xgb_p_team2_win
0,77,Round of 32,2026-06-30,France,Sweden,True,0.713083,0.169390,0.117527,2-1,2.68,1.05,0.768806,0.188479,0.042715
1,79,Round of 32,2026-06-30,Mexico,Ecuador,False,0.313606,0.393039,0.293355,0-0,0.76,0.73,0.434231,0.340882,0.224887
2,80,Round of 32,2026-07-01,England,DR Congo,True,0.559336,0.327887,0.112777,1-0,1.21,0.40,0.680517,0.228259,0.091224
3,81,Round of 32,2026-07-01,United States,Bosnia and Herzegovina,False,0.596911,0.225861,0.177228,1-1,2.06,1.05,0.714913,0.190086,0.095001
4,82,Round of 32,2026-07-01,Belgium,Senegal,True,0.481788,0.271557,0.246656,1-1,1.60,1.08,0.565138,0.244370,0.190492
5,83,Round of 32,2026-07-02,Portugal,Croatia,True,0.551732,0.256896,0.191372,1-1,1.74,0.94,0.508314,0.290016,0.201670
6,84,Round of 32,2026-07-02,Spain,Austria,True,0.604414,0.247257,0.148329,1-0,1.79,0.78,0.658169,0.260072,0.081759
7,85,Round of 32,2026-07-02,Switzerland,Algeria,True,0.460662,0.269572,0.269766,1-1,1.60,1.18,0.545993,0.268504,0.185503
8,86,Round of 32,2026-07-03,Argentina,Cape Verde,True,0.825592,0.145605,0.028803,2-0,2.27,0.28,0.816435,0.102158,0.081407
9,87,Round of 32,2026-07-03,Colombia,Ghana,True,0.718507,0.201368,0.080125,2-0,2.04,0.55,0.768206,0.161211,0.070583


## 6. Monte Carlo simulation of the remaining bracket (10,000 runs)

For every still-undetermined match we draw a scoreline from that pairing's Dixon-Coles grid.
Already-played knockout matches are fixed to their real result. Drawn knockout matches are
resolved with a simple penalty-shootout model. Winners propagate up the bracket (`W74` = "the
winner of match #74", `L101` = "the loser of match #101", matching the official 2026 bracket
numbering) until we reach the Final.

**Simplifying assumption (tune freely):** team strength (Elo / Dixon-Coles ratings) is held
fixed for the duration of a single simulated run — we don't re-fit ratings mid-bracket. This
is standard practice for tournament Monte Carlo sims and keeps 10,000 runs fast; the cost is
that it can't capture e.g. a key injury suffered in the Round of 32 affecting the quarter-final.


In [14]:
N_SIMS = 10000

def load_bracket(wc_json):
    ko = [m for m in wc_json["matches"] if "num" in m]
    ko.sort(key=lambda m: m["num"])
    bracket = {}
    for m in ko:
        t1, t2 = m["team1"], m["team2"]
        t1 = t1 if is_placeholder(t1) else canon(t1)
        t2 = t2 if is_placeholder(t2) else canon(t2)
        s = m.get("score")
        winner = loser = None
        if s and s.get("ft") is not None:
            h, a = s["ft"]
            if "p" in s:
                winner = t1 if s["p"][0] > s["p"][1] else t2
            elif "et" in s and s["et"][0] != s["et"][1]:
                winner = t1 if s["et"][0] > s["et"][1] else t2
            elif h != a:
                winner = t1 if h > a else t2
            if winner:
                loser = t2 if winner == t1 else t1
        bracket[m["num"]] = {
            "round": m["round"], "date": m["date"], "team1": t1, "team2": t2,
            "ground": m.get("ground"), "played": s is not None and s.get("ft") is not None,
            "winner": winner, "loser": loser,
        }
    return bracket

def resolve_placeholder(token, results):
    kind, num = token[0], int(token[1:])
    return results[num]["winner"] if kind == "W" else results[num]["loser"]

def is_neutral_venue(team, ground):
    return GROUND_COUNTRY.get(ground) != team

def penalty_shootout_winner(team_a, team_b, elo_a, elo_b, rng):
    # ASSUMPTION: penalties are close to a coin flip; we damp the Elo gap (divide by 600
    # instead of the usual 400) to reflect that shootouts are only weakly skill-correlated.
    p_a = 1.0 / (1.0 + 10 ** (-(elo_a - elo_b) / 600.0))
    return team_a if rng.random() < p_a else team_b

def simulate_bracket_once(bracket, dc_model, elo_now, rng):
    results = {}
    for num in sorted(bracket.keys()):
        m = bracket[num]
        if m["played"]:
            results[num] = {"winner": m["winner"], "loser": m["loser"], "team1": m["team1"], "team2": m["team2"]}
            continue
        t1 = m["team1"] if not is_placeholder(m["team1"]) else resolve_placeholder(m["team1"], results)
        t2 = m["team2"] if not is_placeholder(m["team2"]) else resolve_placeholder(m["team2"], results)
        neutral = is_neutral_venue(t1, m["ground"])
        grid, lam, mu = dc_score_grid(dc_model, t1, t2, neutral)
        flat = grid.flatten()
        idx = rng.choice(len(flat), p=flat / flat.sum())
        i, j = np.unravel_index(idx, grid.shape)
        if i > j:
            winner, loser = t1, t2
        elif j > i:
            winner, loser = t2, t1
        else:
            winner = penalty_shootout_winner(t1, t2, elo_now.get(t1, 1500.0), elo_now.get(t2, 1500.0), rng)
            loser = t2 if winner == t1 else t1
        results[num] = {"winner": winner, "loser": loser, "team1": t1, "team2": t2}
    return results

def monte_carlo(bracket, dc_model, elo_now, n_sims=N_SIMS, seed=42):
    rng = np.random.default_rng(seed)
    nums = sorted(bracket.keys())
    ROUND_LABEL = {"Round of 32": "R32", "Round of 16": "R16", "Quarter-final": "QF",
                   "Semi-final": "SF", "Final": "Final"}
    all_teams = set()
    for n in nums:
        for t in (bracket[n]["team1"], bracket[n]["team2"]):
            if not is_placeholder(t):
                all_teams.add(t)
    progress = {t: {"R32": 0, "R16": 0, "QF": 0, "SF": 0, "Final": 0, "Champion": 0} for t in all_teams}
    final_num = max(n for n in nums if bracket[n]["round"] == "Final")

    t0 = time.time()
    for s in range(n_sims):
        res = simulate_bracket_once(bracket, dc_model, elo_now, rng)
        for n in nums:
            rnd = ROUND_LABEL.get(bracket[n]["round"])
            if rnd is None:
                continue
            for side in ("team1", "team2"):
                t = res[n].get(side)
                if t and t in progress:
                    progress[t][rnd] += 1
        champ = res[final_num]["winner"]
        if champ in progress:
            progress[champ]["Champion"] += 1
        if (s + 1) % 2000 == 0:
            print(f"  ...{s+1}/{n_sims} sims ({time.time()-t0:.0f}s elapsed)")

    out = pd.DataFrame(progress).T / n_sims
    return out.sort_values("Champion", ascending=False)

bracket = load_bracket(wc2026)
elo_now = dict(history_state.elo)
print(f"Running {N_SIMS} simulations of the remaining bracket...")
bracket_probs = monte_carlo(bracket, dc_model_today, elo_now, n_sims=N_SIMS)
bracket_probs.head(20)


Running 10000 simulations of the remaining bracket...
  ...2000/10000 sims (12s elapsed)
  ...4000/10000 sims (23s elapsed)
  ...6000/10000 sims (37s elapsed)
  ...8000/10000 sims (68s elapsed)
  ...10000/10000 sims (110s elapsed)


,R32,R16,QF,SF,Final,Champion
Argentina,1.0,0.9559,0.8171,0.6064,0.4211,0.2815
Spain,1.0,0.7933,0.5239,0.4088,0.2696,0.1460
France,1.0,0.8459,0.6585,0.4354,0.2425,0.1200
Brazil,1.0,1.0000,0.6547,0.3783,0.1695,0.0915
England,1.0,0.7992,0.5136,0.2962,0.1371,0.0744
Portugal,1.0,0.7088,0.3246,0.2290,0.1277,0.0580
Colombia,1.0,0.8869,0.5735,0.2247,0.1158,0.0558
Morocco,1.0,1.0000,0.6791,0.3315,0.1485,0.0535
Belgium,1.0,0.6455,0.4870,0.1823,0.0867,0.0314
Norway,1.0,1.0000,0.3453,0.1497,0.0459,0.0182


## 7. Output: clean prediction tables + CSV export

In [15]:
# --- 7a. Match-level predictions for every fixture with two known teams (Section 5 output) ---
match_predictions = named_predictions.copy()
match_predictions["matchup"] = match_predictions["team1"] + " vs " + match_predictions["team2"]
match_predictions = match_predictions[[
    "match_num", "round", "date", "matchup",
    "dc_p_team1_win", "dc_p_draw_reg", "dc_p_team2_win", "dc_predicted_scoreline",
    "xgb_p_team1_win", "xgb_p_draw", "xgb_p_team2_win",
]].round(3)
match_predictions.to_csv("wc2026_match_predictions.csv", index=False)
print("Saved wc2026_match_predictions.csv")
match_predictions


Saved wc2026_match_predictions.csv


,match_num,round,date,matchup,dc_p_team1_win,dc_p_draw_reg,dc_p_team2_win,dc_predicted_scoreline,xgb_p_team1_win,xgb_p_draw,xgb_p_team2_win
0,77,Round of 32,2026-06-30,France vs Sweden,0.713,0.169,0.118,2-1,0.769,0.188,0.043
1,79,Round of 32,2026-06-30,Mexico vs Ecuador,0.314,0.393,0.293,0-0,0.434,0.341,0.225
2,80,Round of 32,2026-07-01,England vs DR Congo,0.559,0.328,0.113,1-0,0.681,0.228,0.091
3,81,Round of 32,2026-07-01,United States vs Bosnia and Herzegovina,0.597,0.226,0.177,1-1,0.715,0.190,0.095
4,82,Round of 32,2026-07-01,Belgium vs Senegal,0.482,0.272,0.247,1-1,0.565,0.244,0.190
5,83,Round of 32,2026-07-02,Portugal vs Croatia,0.552,0.257,0.191,1-1,0.508,0.290,0.202
6,84,Round of 32,2026-07-02,Spain vs Austria,0.604,0.247,0.148,1-0,0.658,0.260,0.082
7,85,Round of 32,2026-07-02,Switzerland vs Algeria,0.461,0.270,0.270,1-1,0.546,0.269,0.186
8,86,Round of 32,2026-07-03,Argentina vs Cape Verde,0.826,0.146,0.029,2-0,0.816,0.102,0.081
9,87,Round of 32,2026-07-03,Colombia vs Ghana,0.719,0.201,0.080,2-0,0.768,0.161,0.071


In [16]:
# --- 7b. Tournament-level probabilities for every team still alive (Section 6 output) ---
bracket_probs_out = bracket_probs.round(4).reset_index().rename(columns={"index": "team"})
bracket_probs_out.to_csv("wc2026_bracket_probabilities.csv", index=False)
print("Saved wc2026_bracket_probabilities.csv")
bracket_probs_out


Saved wc2026_bracket_probabilities.csv


,team,R32,R16,QF,SF,Final,Champion
0,Argentina,1.0,0.9559,0.8171,0.6064,0.4211,0.2815
1,Spain,1.0,0.7933,0.5239,0.4088,0.2696,0.1460
2,France,1.0,0.8459,0.6585,0.4354,0.2425,0.1200
3,Brazil,1.0,1.0000,0.6547,0.3783,0.1695,0.0915
4,England,1.0,0.7992,0.5136,0.2962,0.1371,0.0744
5,Portugal,1.0,0.7088,0.3246,0.2290,0.1277,0.0580
6,Colombia,1.0,0.8869,0.5735,0.2247,0.1158,0.0558
7,Morocco,1.0,1.0000,0.6791,0.3315,0.1485,0.0535
8,Belgium,1.0,0.6455,0.4870,0.1823,0.0867,0.0314
9,Norway,1.0,1.0000,0.3453,0.1497,0.0459,0.0182


In [19]:
# --- 7c. Quick sanity chart: title-win probability for the top contenders ---
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

top = bracket_probs_out.head(12)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top["team"][::-1], top["Champion"][::-1])
ax.set_xlabel("Probability of winning the 2026 World Cup")
ax.set_title(f"Monte Carlo simulation ({N_SIMS:,} runs) — title odds")
plt.tight_layout()
plt.savefig("wc2026_title_odds.png", dpi=120)
plt.show()
print("Saved wc2026_title_odds.png")


Saved wc2026_title_odds.png


---
### Where to tune things later

| Assumption | Where | Default |
|---|---|---|
| Elo start rating / home bonus / K-factors | Section 2 | 1500 / 100 / 30-60 |
| Recent-form window | Section 2 | last 10 matches |
| Head-to-head window | Section 2 | last 5 meetings |
| Dixon-Coles time-decay (`XI`) | Section 3 | 0.0015/day |
| Dixon-Coles training window | Section 3 | 10 years |
| Minimum matches for a team-specific rating | Section 3 | 10 |
| XGBoost hyperparameters | Section 3 | 300 trees, depth 4, lr 0.05 |
| Penalty shootout model | Section 6 | Elo-gap/600 logistic |
| Monte Carlo iterations | Section 6 | 10,000 |
